# Qwen2.5-Coder 3B on Google Colab Free Tier

A one-model, 4-bit GPU setup for using Qwen2.5-Coder 3B as a third backend beside the local 0.5B and 1.5B models. It uses a public Hugging Face model and requires no GCP project or billing account. Colab may require a Google account and free GPU availability is not guaranteed.

**Connect from VS Code:** Open or create this notebook, choose **Select Kernel → Colab → Auto Connect**, and sign in when prompted. If Colab offers runtime choices, select a T4 GPU when available. Run the Drive mount cell and approve access when prompted. The model files are saved under `MyDrive/colab-models` and reused in later sessions; the runtime, installed packages, API key, and tunnel are still temporary. The model download needs about 6 GB of Drive storage. Free GPU availability, session duration, and quotas vary. The optional authenticated tunnel makes the Colab model reachable by your local workspace API; task text sent through that route leaves your computer and is processed by the Colab runtime and Cloudflare tunnel. Do not send secrets or sensitive task details. Keep the generated API key private and clear notebook outputs before sharing the notebook. The notebook does not connect to or modify the workspace task database.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU is attached. In Colab, choose Runtime > Change runtime type > T4 GPU, "
        "then reconnect and run this cell again."
    )

gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} ({gpu.total_memory / 1024**3:.1f} GiB VRAM)")
print(f"CUDA: {torch.version.cuda}")

In [ ]:
%pip -q install "transformers>=4.45,<5" accelerate bitsandbytes fastapi uvicorn

## Mount Drive for persistent model files

Run this cell in the Colab UI and approve the Google Drive prompt. Only the Hugging Face model snapshot is stored in Drive; the live runtime and endpoint are recreated each session. The first download is about 6 GB.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"
MODEL_DIR = Path("/content/drive/MyDrive/colab-models/Qwen2.5-Coder-3B-Instruct")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"Persistent model directory: {MODEL_DIR}")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import snapshot_download

LOAD_IN_4BIT = True

snapshot_download(repo_id=MODEL_ID, local_dir=str(MODEL_DIR))
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
quantization = BitsAndBytesConfig(
    load_in_4bit=LOAD_IN_4BIT,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(
    str(MODEL_DIR),
    quantization_config=quantization,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()
print(f"Loaded {MODEL_ID} in 4-bit mode on {model.device}.")

## Start the authenticated workspace-compatible endpoint

Run these cells after loading the model. The endpoint accepts the same OpenAI-compatible chat-completion request used by the workspace API. A random bearer key protects generation requests; the health check is intentionally public and reveals only that the runtime is awake. The temporary Cloudflare quick tunnel has a public URL, so keep the bearer key private.

In [ ]:
import asyncio
import hmac
import secrets
import threading
import time
import torch
import uvicorn
from fastapi import FastAPI, Header, HTTPException, Request

WORKSPACE_API_KEY = secrets.token_urlsafe(32)
workspace_api = FastAPI()
inference_lock = threading.Lock()

@workspace_api.get("/health")
async def health():
    return {"status": "ok"}

@workspace_api.post("/v1/chat/completions")
async def chat_completions(request: Request, authorization: str | None = Header(default=None)):
    expected_authorization = "Bear" + "er " + WORKSPACE_API_KEY
    if not hmac.compare_digest(authorization or "", expected_authorization):
        raise HTTPException(status_code=401, detail="Invalid bearer token")
    payload = await request.json()
    messages = payload.get("messages")
    if not isinstance(messages, list) or not messages:
        raise HTTPException(status_code=400, detail="messages must be a non-empty list")
    max_new_tokens = payload.get("max_tokens", 220)
    if not isinstance(max_new_tokens, int) or not 1 <= max_new_tokens <= 300:
        raise HTTPException(status_code=400, detail="max_tokens must be between 1 and 300")
    if not inference_lock.acquire(blocking=False):
        raise HTTPException(status_code=503, detail="A model turn is already running")
    try:
        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        ).to(model.device)
        if inputs["input_ids"].shape[-1] > 2048:
            raise HTTPException(status_code=400, detail="Prompt exceeds the 2048-token limit")
        with torch.inference_mode():
            output = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=float(payload.get("temperature", 0.2)) > 0,
                temperature=max(float(payload.get("temperature", 0.2)), 1e-5),
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        content = tokenizer.decode(
            output[0, inputs["input_ids"].shape[-1]:],
            skip_special_tokens=True,
        ).strip()
        return {
            "choices": [{"message": {"role": "assistant", "content": content}}],
            "model": MODEL_ID,
        }
    finally:
        inference_lock.release()

def run_workspace_api():
    uvicorn.run(workspace_api, host="127.0.0.1", port=8000, log_level="warning")

api_thread = threading.Thread(target=run_workspace_api, daemon=True)
api_thread.start()
for attempt in range(30):
    try:
        import urllib.request
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=1) as response:
            if response.status == 200:
                break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Workspace-compatible model API did not start")
print("Authenticated model API is ready on port 8000.")

In [ ]:
import os
import hashlib
import platform
import subprocess
import time
import urllib.request

if platform.system() != "Linux" or platform.machine() not in ("x86_64", "amd64"):
    raise RuntimeError("This quick-tunnel cell currently supports Colab Linux x86_64 runtimes only.")

cloudflared_path = "/tmp/cloudflared"
cloudflared_version = "2026.10.7"
cloudflared_asset_url = "https://api.github.com/repos/cloudflare/cloudflared/releases/assets/692817008"
cloudflared_sha256 = "1f99fb518b83dec2ba8d870a7cc991d82a77e6196295a3d2754ba17ab9f91872"
cloudflared_request = urllib.request.Request(
    cloudflared_asset_url,
    headers={"Accept": "application/octet-stream", "User-Agent": "workspace-colab-setup"},
)
digest = hashlib.sha256()
with urllib.request.urlopen(cloudflared_request, timeout=120) as response, open(cloudflared_path, "wb") as binary:
    while chunk := response.read(1024 * 1024):
        digest.update(chunk)
        binary.write(chunk)
if digest.hexdigest() != cloudflared_sha256:
    raise RuntimeError(f"cloudflared {cloudflared_version} failed SHA-256 verification")
os.chmod(cloudflared_path, 0o700)
tunnel = subprocess.Popen(
    [cloudflared_path, "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
public_url = None
deadline = time.time() + 90
while time.time() < deadline:
    line = tunnel.stdout.readline()
    if not line and tunnel.poll() is not None:
        raise RuntimeError(f"cloudflared exited with code {tunnel.returncode}")
    if "trycloudflare.com" in line:
        import re
        match = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            break
if not public_url:
    tunnel.terminate()
    raise TimeoutError("Could not obtain a Cloudflare quick-tunnel URL")

with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=10) as response:
    if response.status != 200:
        raise RuntimeError(f"Local model API health check returned HTTP {response.status}")

print("Temporary authenticated Colab endpoint (expires when this runtime stops):")
print(f"LOCAL_AI_URL={public_url}")
print("In a local terminal, set the endpoint and enter the key without putting it in shell history:")
print(f"export LOCAL_AI_URL='{public_url}'")
print("read -rsp 'Colab bearer key: ' LOCAL_AI_API_KEY; export LOCAL_AI_API_KEY; echo")
print("./workspace dev-colab")
print("Colab bearer key (keep private; clear notebook output before sharing):")
print(WORKSPACE_API_KEY)

## Run a task-refinement turn

Edit the task fields as needed. This mirrors the local workspace assistant's JSON output shape. The input is kept short and the output is capped to avoid wasting the free runtime.

In [ ]:
import json
import time

task = {
    "title": "Add validation to the task creation API",
    "project": "api",
    "task_type": "feature",
    "description": "Reject invalid task input and return clear errors.",
    "acceptance_criteria": "Invalid titles and unsupported fields return 400; valid tasks still persist."
}

messages = [
    {
        "role": "system",
        "content": (
            "Refine this software task. Return only a JSON object with string fields "
            "description and acceptance_criteria. The acceptance_criteria value must be one string, "
            "never an array or object. Preserve intent; do not invent requirements or claim work is done. "
            "Make criteria concise and verifiable."
        ),
    },
    {"role": "user", "content": json.dumps(task, ensure_ascii=False)},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(model.device)
if inputs["input_ids"].shape[-1] > 2048:
    raise ValueError("Prompt exceeds the 2048-token evaluation budget; shorten the task fields.")

started = time.perf_counter()
with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=220,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
    )
elapsed = time.perf_counter() - started
generated_tokens = output.shape[-1] - inputs["input_ids"].shape[-1]
answer = tokenizer.decode(
    output[0, inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
).strip()

print(f"Elapsed: {elapsed:.2f}s | Generated tokens: {generated_tokens} | Speed: {generated_tokens / max(elapsed, 1e-6):.2f} tokens/s")
print(answer)

In [ ]:
try:
    json_answer = answer.strip()
    if json_answer.startswith("```"):
        json_answer = json_answer.removeprefix("```json").removeprefix("```")
        json_answer = json_answer.removesuffix("```").strip()
    draft = json.loads(json_answer)
    if not isinstance(draft, dict):
        raise ValueError("The model response must be a JSON object")
    criteria = draft.get("acceptance_criteria")
    if isinstance(criteria, list):
        formatted = []
        for index, item in enumerate(criteria, start=1):
            if isinstance(item, str):
                details = item.strip()
            elif isinstance(item, dict) and all(isinstance(value, str) for value in item.values()):
                details = "; ".join(
                    f"{field.replace('_', ' ')}: {value.strip()}"
                    for field, value in item.items()
                    if value.strip()
                )
            else:
                raise ValueError("Invalid acceptance criterion")
            if not details:
                raise ValueError("Empty acceptance criterion")
            formatted.append(f"{index}. {details}")
        draft["acceptance_criteria"] = "\n".join(formatted)
    assert isinstance(draft.get("description"), str)
    assert isinstance(draft.get("acceptance_criteria"), str)
    print("Output check: valid JSON with description and acceptance_criteria.")
except (json.JSONDecodeError, AssertionError, ValueError):
    print("Output check: the model did not return the expected JSON shape; review or retry manually.")

In [ ]:
from google.colab import drive

drive.mount('/content/drive')